# Drum: konteks maksimal 90 detik lalu stitching

Eksperimen pembanding setelah drum full-song satu konteks ditolak. Pakai **VM Colab aktif yang memiliki runtime/bobot ACE-Step**; salin sel ini ke notebook aktif agar cache tetap dipakai. Helper tersemat; tidak perlu ZIP tools. Piano asli dipakai kembali jika tersedia.

Setting model tetap sama: acoustic drum caption, Base / LEGO / CUDA FP32 / seed 0 / CFG 7 / 50 langkah. Batas **90 detik termasuk margin enam detik dan padding model**, bukan core 90 detik ditambah margin. Untuk piano 4:03: tiga konteks model 90/89/90 detik dan dua sambungan. Crossfade float equal-power .58 detik; kualitas belum terbukti. Groove/timbre antar-chunk masih dapat berubah.

Tidak mengulang bass atau mengganti baseline 30 detik. DRY_RUN=True hanya membuat plan. Notebook ini tidak memasang atau mengunduh ulang model. VM kosong perlu sel setup/runtime/bobot notebook 12 terlebih dahulu, tanpa menjalankan generasi bassnya.


In [ ]:
from pathlib import Path
import base64, gzip, hashlib, importlib.util, json, shutil, sys, tempfile, uuid, zipfile
from google.colab import files
DRY_RUN = False
if not DRY_RUN:
    assert shutil.which("nvidia-smi"), "Aktifkan GPU dan gunakan VM dengan runtime/bobot ACE-Step yang sudah siap"
for dependency in ("numpy", "scipy", "soundfile"):
    assert importlib.util.find_spec(dependency), f"Kernel notebook memerlukan {dependency}"
# BEGIN EMBEDDED CONTEXT HELPERS
EMBEDDED_HELPERS = {
  "electric_full_song.py": {
    "sha256": "5629f87436e2168a80fe5e34327d41a12ebc3ffc1ac23d914c54ba21d2c55c25",
    "gzip_base64": "H4sIAAAAAAAC/70863Lbxnr/9RRbnB8BFQqk7DpnyoSZUWQpVo8taSw77VTVYEBgSW4EAji4UKZVzfQd+oZ9kn6XXWABgpTipvVkYmLv+91va8dxblSyiOVRFhSFOB7/93/+1w/jcSHOYhmWuQrFL9A+FCoJ4yqCkaIqpAgKEYg4TaSosjgNIhmJ0zQOZmIp40zm3sHB50IWolxKIb+oosR5mUoSGJdXSalWsEQSCZksFKxxcn0xFA+qXKZVKVRZiEgFiySFaaE4fX/xXXFwPB69hjOFQeaJK5hRpFUeShGmSSm/lEOBJ1mlkYxhSBz/KJIUhsCe4bJK7nHzNBewXBku4cM7cBznQK2yNC9FkC+yIC+k+Q7TbGN+/w5rmN+roFya32lxMM/TlcigLVYzoZuvrSFFNcvyNJRFUbds6p94f16h3GQAJt18A3/H8jJYySILwvpEVaUi8/shWNftX1U2V7E8OPh48ulMTMXrV+Px+OD649nN2Sf4fDwQ8McpwqVcBf5a5oVKE2cijofCURH8cKTG8NEMMOxAM0EQe2D3opTZ0fr4DXZKZ8iLRXKtQokjwioKcEqWy1DplZ05kEL5+hW2q6Qo82olkxI7zAaFlFEBDbfjO73iAi4XJKH0C0AcrvxXb0zT5zKX1A7nwClvsDmd4xaRX6Z+mFXQ+imvpF4JaKPU57gogBZLoLU5YBsuXm5ioENDz3gYAfuWQS6CMExXWZAoPKknTkSUb4YiCcoqD2KRxVV4D8uUSF4ObWP9cZBiRRjLII83QEdArRVvG5RlEN4D0xRLRFSSllLksH+APIF0HwAnzWJkHmAWsQzyVZqosPB6NjkRBZGFWFWFAhjR8WPkGpUADyZpvtKNR/pOuVwAx8G68zSO0wekfuTDTAVJ+l3RswXvD/cGusTReMJokwQrPBKg/ung4CCSc+DKIkyBkPw8TUsX6WIoAP+AqVxG00sA0mDCuHCcc4XXFLNcRQvg06UM75G5DZMjSGZpei8WcToL4gL5EyipAiJdS4AmUEOS4FkMU8MKcBaHD49bA4Ujv9ExBl4uAeVr6Q6ofxXk94B2GOI6ICLmalGMAECyLIiaPeRrJMcyTeNiBLBa+XIdxCMQTV62IUINc5WV0CfLKvO1yMI+3gChARNU5KII0LfGPzmMzwEvcexSlxgBMa3ggKrwkVndAWAlpzbEnz4oL6rmDTSFKhBGAoHarE4r6nvXQwee/AIUHIFYzt0uJPS6uFTveenMgQJwfuQ7nuV5mrtz5+PV1SexCQADkcpUrJZiVt0HSYNJxuxEPOKKT86gCwJspjbAZAQ8XkrEB8BTnwzBAAQaAZ0CICIVlh5KxHu5KVxNW4QrEM7AmgPrzDiTIAHzXF5iaJYaAepINTh20w1guPhVJjIPUBMdfUCiKrRQ2/pzyBM9JE3XQTWyCjIfBZ5/ONKrD4adYSAxfa2NfB7jHzqDDqRVm2os0HhBxK0N/hhMuQyiDQDulm5sX72Z3L4GbGJIz1nLJErz0cnp2dENyvNj783Ig8b1aKaSUbYBZkwcizjbK6Ec2LsUEUOWqgQ4pasyRsS5XhHMZSmTIs0Le6M7Jo1lCuqE6ALkpIxcvi1csrlczRuxTFw9YSB+FscW020TsHMSgfCRM8B4FnSJ1hMXhRJI4f67i8tPwM3JAki7COJgCf8vKwNjNdmWlvaf7wXKCu93gIALNOKCyhuaSw1qBBI76FZQfHgXc3GgQkk8riUswcxnvLgoZTUBcQvACdv2ItaBXte50eKL54GAkA5umxYeSZ7pVDigmHl7xyaFA0tg6Mk1zvYDXH9pgT2TcbUSASAhw//99gGh6Yl/BhAnKEYKGEOytTYHwQoQs3QG++4FuROpWnkUo+MxMB2Ynb5R7v68imMfedZT2SaZDUUcxJUAvQyWpliQCCiU57RwwxfVKCDdvbSBD4abhwRxOwHkAScCll2Dh44GQblgd2s14gC98+rzHK07WwqTEYFmnZdmQOE4m3qBsPKZMyBLG6yFdGLLc5eavIUsaUEQa4Cfoahbk3AJ+hPQ22otglX2oCJQHoOB+AdQjmg4DsUr+K9XJfwGoDPoPfuSAYRBMaFlIVMwNu/ffRXXpx+Of9iW/c059H0H+vrA1mjogAz8Kn1wHxYuy0owPcAOlyVCDwypKdh6UYXCOk2mx/CxCFQyHXt/HYLG/OLXXeCnNBYHnDBWoSrRLVkcaVkM3gfb0AUcPU3AsphtwBoC7fqQour9EcTKIgg3IgbTBxwP2H4DpAokUNbWBhnqQASIZMK3sddz+fdK5ZIaaeQaAUZmB12juQTfoH36mtfcINm4YEODxYyWsLsGzQcrDlAOIh+2u1w4Heg2tLQH9RB0ToBRwdxVQAlrNjLWqCX4SG3BTr4QnE/8JMZmhTF8mJOJn6atk+I3gNoeitfB5uMtqWATDfiAgs/EG/5sI/anqb3HT4hM0ji0NKr28fD4TtNW4015IC/cWxvyrjOfrzK5QN5zjtBrBC8V5fLRGv8v8Sj0yY3KYRZlwhv02wDOEfkrNA4PTkuXuqVGIDYGc7Q0nHUK8k5OH/HwT86uRYNcL4GMx9NDPBR5TEfhhD2qcOUXxz/EUo9lzhjcDVmNTdHlYagAkVZg+0+NVNEjQfDg+oa8glnh6pFHNfBJg+qBezA4d97KMEXvvqaFlSqA2sLlhMgfdX4kHk3vExA6UMgj7/fUFrPcqCUBmK9o2dX8D9o5q3r53/Ir2OnOwQdJkB0MHxbL4NWbH4biIQdK88lp/2N8C1CymEzz7i4e1N0v4EMmnRa7PcctQZdfGpVcY8ByDdhEsw7XCJxdx29GtG6wT+fWUmP7ig096e7jMYqFjuR4Vki0kE3CCz3Wh2UaowCJkT7yBjYmRgUWBIj5qNAwSnO1UEkQ+8ugQDeJycIwOoMxmafQgyTiYSChcC3JQuzlMyFuixgYNZMsYzpyBZz8Bx98lFzJnT6FM0c/vZyai9LEFGWHo51RSyYBq6PuYk7XrL4OFNg5Mbq9hDEXr3Krl3Xubp165bsB8DmRj23MtdFWL0d4axb/CfEHLa5NbDWhNeN+Rqx64zd7EXtDRtkWtkQ6ZwPhR0IyGmyEYzgiaGfYCQ4LQhxPslJJBS7AC3igOdr3Ag9mabO9R+QwIqxGkRqJdAd2KJ6rkW7ySyjzrDRURvThre4jhZYaOqcFYWrIkU4/vZ+eB2BY83AMD6HhztPQJjRk6oHRxwZGmK5WeAmwLf9ExbatxO72gXFSH+N7OEePtrtrnZUHkfYzyg/sMzrIN2k5gpPeo6Pt9Zbb+o986Ub98RI20QMJHZKKQznEo7Upc/ic5qsVH5EqBt9KJM4domcvaYL+5b1Z3Xa0MFLrS86hqXCLVms61UdhF9LwGpiivPf3Zu9jsA8sAwHcMwrqTzsTD814ms5o4bCGDkI1BM3tDTn3eDSMGsulMUww7Ay09sDhD3o4nlJGdiAMvz3wIIEDg1Xhuq/Ij2GHBiPFl1eXZ0hniAoM8eZATQijRtE9LFHozOI0vBeTaX0iD2MQmqKIgCdmNzIvuCcPHlya2YqyaVjunOD8+9gBwLoG5ofiH/V5EOYKKYJR8aij9r+1gvYa0/D12Ej7SRtvGLpEXod2+vtphyk6k0FJgXjQNU6UPiSthhAILNJf/fMfghzPSmOcE2SRI/Ku7ItQFkYlYKyhFOXY8uhebka0vFjJMoiCMvCcuyemm9p8cxvishc0cVu7bbB7qjYxIzPtsT8XshOU9c2bQxQE/BPMMVhU/zS0hvhsdtQDW1YIE/ZT39Jwl2aqnrQPDIMnze8ajEQ2tQlUr9QyivCcyIlbG2n+3MK1vuXW+A6bYmAhlj6GIWAMM6FDgsNn8odW+uzZwQK/LSVNNsrfQ+kyiXzNS80+uqHvLjZjNPwkI3uPxjw2d0eXDjpAu+0hU5WgUaAJzaCk5f2YRu3/yC8yBNvG5zAEBYwaJ+iQ8iog3cgextyrSaZNOZXWRD0+SkzHosECg3SIRmdVwUCB73jDqVWhVeoRiK0AMyqULRudX79+NdJBEszt5CqyEi18Ckw5cFaRUxTYZOxCCiFyW+1K1adl3wNc+ToBGEEHHB9tTJ6EUSLXzhZSZKr53qca/4VgZ+9Haprc044Nx5tpFWmH+Pb5lewQVBksL4NVv3+pw5dg/69VniZ0CuNmbnfRPHvotG+UfTIrWeQ67879d59/8U9PTt+RcoPvfz371Hx/uH7/y8np384u3+qv06vL84tf3158dCxLPS08vdktLo0xTWtzbjtoD/SqDIPxrj7A1fn5+4vLs6lzDPt8+nhyeXN+9fHD2ccbu0drthYM3T3xazxygKmVVtDVA+zKvET9bgdW++ZrXWpy7GkeLm1jkBo8TONgSLt2GJ6La59+fnsiqqRxMDBtRzFujTlSc6fXn0cfrm/EPIjjWRDeO62z6NyIB/ZaFAPFYkwXJIv+tEfiXLhaYbfBPjiDufpAexWYwz7ndPs7Xsa1lvROQolw0V3WFVH2+JRDX4Fc81mkIiX7TTLHLWQ8BykU5AsQs4eH9w/4azuXxe3MwboygLiXiwMmveC0a020KNE+TyEQ1E47EoEn8SKskgAqZQzqIoPWsCzHYzu316Ya4U7YG71Vn0boUY9+OzmbNkUK87gqlpY/0YliF1VGGdVnwdUFVBvYAB5/HUifbqFBy9DSoS2z3/b1dmMUoNHGfjsCoskIRhmCwvm/8G+LgkyNj+5xuwt0CcfH8IHvv4hAavCZSTvB1Do6G/SUKgbU85ePoGsNpxAim/77T4F/uOO2rhBBice0Z7VtzdJY6Rxs/y26Z8aN6EuXLWzhAgZ0cEAj/yJOQZCgM6DtBBfcoHWqwDEz9Urf2VVSbHxTkdQR20ec0PC0wkHJ4dnJF9h3KxdjND4ZzS0PzzSyUbDQifS0HddomrvDdLCkLzwClofMlVaF+pxZHCSu2VIrBdsGGmL5UHe60VCYU/g9nRkrEbMKht382hyYmvIlZDDi6amhBy6B2uH2AJm2CpB03KddwzTFLBWsMnUKEkF+RhZb13a0oAjAsy6ijcimRRstQGwAo06NmGs063S/ZtWkN9XAbIQXW5j1bXcDRHABDcjBRM0BivWGewpstJjT0D7oAFFDDyho2gBjWFPgtCGBPqTWq3Whj/VjyNVVDjKXgif51HKkuEfjo8w3jbjQ5Gc4jvm8iwYMw2Ql1kTKM/qJCETtjOrNWmuLMMHrKKti6szBjgBOGvKM6dx5JL1AXygs0fzy/SdwH6mpVVmD+pTNQpRIsXX2byQs44ugneFTpGsrIRumFbIcZWCP7TzMizIr/x+JuzoJF4AYXU3RK2SHdPpIh38adrJz+2OU1rp/OCOn5t1cHJhDdIh9fswVU2cEzi/VXSDn5Wupq2eDsKyVEK/OKzoGfSAH7oOF1N7oH0YRuhhtpJOz4dbloX6M5YQJhTqGTdWoD+dcK/ngrbLXiELXWakvVH9hRuJ3a5Clm/8PCKNm8r2p3RZmYzWDg8UAU1ppxq3H//TqvrvmbpyDlAK4omzmX1QtU7ht/01XYWjI1kUqoJKKKi69rypzEII1bNFf3gJ4Gw8taA/23Nbeby5lhLpge0cbUb24s5BH8QRdfuz9m8rOsT6oBhXf1/lCkTqOvGJqrx5/ce2/PTt/D4z3lstbwOxVa9m22+ZUEBqDHFuz83uo4YxmJ4IS7GY9keOszf5m2rD+ZTiljaCJHZ5xbW1dF4gaFTRqxXcsU2e0JV9bt7BHFgyHvkXG47F/Qjj3sUJ6PNJkYYZqpTXSf+sAoCkpIoVSY7YtBhzH+ZuUmQhEsQKNITDUi7qXPFkGhwjmmEy4uvowwqVg/aYWfxUUVD2PpTFhUIFmryND3SzADhprFZShP09WYOE+R0049k8hIhM96cN9t2yyRcJWteJOUmNSp9EaGxqrUkf0uuUNh7sqHIbG0tfWsSbOnkUwVRNkLgitGZbXTXQ5sFX5C3zdN3Hwx6Jfp1cfPlx8uhnuLa/4hvAXayYyWn2tAqwp4IksFjJq7VDH47AKPEJL0KYqHYi3kMUaFsnoMi3PQVdGpppF53tx2hzbwcri6cbMYquXVrKqLY2J23owwVxoVW6biVatN+b/rGIDexCmmXwMSuhiPI6w7jMTWg9yTPAkUvM5lrrzaxIwGAo6JDhwJsvRKCKtYyMOJv7ZhfGTbsG5CdaZbQfbOLKvRxXnJs/aejikHxHoNz8c1k7zzbcl5LXQoyQbmeSYzGE5D3uZVzCU4bHcN9YFGEKe0AshLwKpij9cxtuW+nWMY+Zj+lrhPM1Qt+SDYZqt41FtZVme9bC2d+UAFb6BKhp3Vye6sGwW74Xvp7A8UiVVWlFwrHlHRRm7SGZgRCMrN2+okBrM8ynx28kZVWuUiuKhoD7IHfHEuyCPHgAF4KCURPzm0QpsUJD1Em7wHRZookRicp3r0kpJiPacPRmWHvXMqOzx5KycWH+dWSuoMRSdak27nhV36LpwGKBIaCKZBdN2vudbb2BxjlYFk54Y59y5fn9yOaEHb49m49vv2rmy7+6eCoNTCk//ev1Z1E+qno988rEOujtfXZ6J65Obm4m9c7PnxHs9f9LVysNnDqcj6EyhzXO9msZ2nNGqlEHrfLtcXtdfk0r0Sdj4vqUZyaE44pwbWfxHOMtpcgs8AHDWsv8HTXSQSIXCYyz4ZD73yR3DWGdDN7Uia6pXQA9OdyZ7tt2KhvxuHT6wCX9Q/LJnd1P2pbXnVnjNUkRuX9huy5YdtJSUTaTWSIr+a0nK0X9jAkUOpvnw2UYz/NYB7V1gjRqMPH5x0XlKIgt9eipBt0L7TEgqWachxx6bc8JOGFTc2vt2fGe9JaI3Qv1xzC273LEwgwY6RVGNTVI/RYLfQ0qOtkCmZ5i8KaljMvJ3pE9wBKx020nB3+GImrW6nc8B1E6K6APxzuzgoM2g5oohqR0LC6Acyqb6C9BH5PDiEnAKWZZUDXJ36/Ag1G99vTzVirZLg2JUyM0TTtaqVnhePx7FRPyw79Xorjej/eUrdlFBDctOvQFuWYL/Q4F81JuxXKTtp61B3HqCurUNGNTgZ/n8RlN9NVuSNYKrg6RD98pqAhcLGLr0G0hwT//6ZjTeoehZBV+9VMFCmq4nmyDx1QEji0jwXm6I+ujBAJmL0DLUn2AwGkR5Ch/TuYNnX4ycsG5vvcag/YCH80Ibrj12K9tZI1Pt1GIiJh8tc+gtMZ0ZX1XhozGrd4ZFAH6hvmqmOu4OQBjlIJjTFfE2jNKFewSp5y73OakJ14q34kIj2lnURG8pL/bed0rifi+/XwaThYdqEH8QtugHPQvk2XRJGgY0ewsCF8EHnwY8lA+AW+NbMNCgmHvENKXFmugRt1ZrSXndw0K+NUzz0ZZE675B6ZtEvMZzWulQx9SX0426KyEph1qQ7l0h7LkI2zx4EQIjwxA2Gbz8TVSjmrCchnXSSOceRDvjYADbLk5vC/VuFdfd1uudbnlYHaeyqtIGezXFy5ds1St3Fm1Vk909C7ALUjL4Nm3BT0rQu7O1uZVDHNgqoklP1I96a+2CTioaeiiwm+P3RVKbc7dq0+52bLV/sZ7eVnx212bNY7bnbogx2K3NOjH256/U78I0NtqwFlXTbYWNKTyiYl9T8VT/vSvabOlgnauzrZytXmdn2DqTwT2/uQZeRNsOjKzZBhzF1oI7R71kYZ1m2b9ue9DuZYkMmRGmW7y0TT9g8CMmd01ox/Z37cn0htWP1vtImFnmIN1N25uhIS+mjenzJPOt3msnEXWw26t8YRa1n3b/FxlUHeDeZe93HSBbYuuprRhW2z3jT/LOLAVv5vWq8W+FNEY7dgX+LalCQK4BjLWxIGPFX8RHIAHFz1zqzGLNmyOW/GzRSDPZ+4NZ6BddRz96h7OYiDyVtqA/ZspcvJN8Qbb2NfW4keQIJMbNfT9KQ/DvrZn47yj4gZ7iNv49Usj0mnIKsGUAjD7dFSPQ/+JEcXt8t3fl+t+ZqNfeO5xjCtY5TFjU8vl37USFy3oymTPNLcZ7Z1rPzezJ+I80TZ2rlSo7b+1MlGwVfFGraoVPZYwFu2MHHaqiWhpCCijkFPO4cKn9M+soTM9EfcSaCG4+X6NHeNMqhdHr0l+4ssm5om+DdYm8gV2T0Sq5pjFIHnbihBoBVd1/hkO2fAKzQ1E/H+MXVPpEkq0dQyLYb5DtdBUz3KNOGdUnGtrLN4cyuSPubMcsubGpXjfZJGrWH4NORM+5IYk6qTny1ljG6Pv+h+C6BOw2J0CWVVgHyDKWnAefSyN99KWJkw/+Bzrm61BTTAAA"
  },
  "drum_probe.py": {
    "sha256": "1ace27dcf94b2ca7846fa6a3aa31114c9a768848a4c2383fd526c513707357dc",
    "gzip_base64": "H4sIAAAAAAAC/7VZW2/bOBZ+969Q80JpVpadBJ1indVDZpouip2dFr3MAhsEBCPRNseypBUlJ2nR/77fIambL21nB+uHxKIOD8/9fIc+Ozt7WTVbr6yKe6m9ovKWTZZNdZGvPK3yVSanpdDaa+jBq9fSK1Wey9S7/vlm+r6Wpffzx5fXXtXktdrK6OzsbKK2ZVHVnqhWpai0bJ9/B8/JsipwlqjXmbr33Iu3eGyJdHMPSRKpdbvySZVLlclJ+ywzmdSVSjjJyY2cQjtRJ5O3727e33zwYu/zxMOHqZQtmEiKRtcqmabQVE935yxk2yKVmXknNbTA4vPpvdASr1K5U4nEu6RJBQsto7KSidKqyLG+zApRX16AVOW6Bk+Z11g23LGopUw1W9zO79zeVaNSkSeS60RkYPwimtPWpaykWcX5oH+OxWJJvFNeFzwpG7b4UDXSMUlEWdvjX+siEzVc0Orl0cneRsHmSVJsS5ErEiny4BdoVwkFh4VeLuqmEpm3qopiJ70HVa+xKdmEns5FJT2Rp16SFRqc12q6FnXoLYssKx56z4u88EpZLYtqSwp58Dlehl65roQJEOKRPuViqxIdeT9nUlRWurWqdejVAsrCcR5cmmlDrUu4wLuX9YOUuWUksZWCEuGYZ08QvPB2BUxnwnMrsyKFzr3pdQQnfZlMJqlcIraSjVhJv2jqsoEGukkomoKFsSIEk7FxFF/CS/cgjhBgzLw0BnHhFv1bla/w3/GZ0caQPbCQ7AujUiTEHe3rt/zlzatfrj/cvAwoGkWVrNVO2jPpA4PBFfAaViG5d2tlMEkXUWIgbMAW+SLTmcpxYru6krmsBHl+Jh9heutZ+7Jjf/AZbtNW2iMM5/M5v+YmbDkF7XwGzZrsO9hTMDaVnLn/LW/LCorslHyItuUlJZp6HK3cLUZ81dJrbdwaKIiU5mRYP1g4S0YPlarlPmHY7Zg4Vs7b/RFf86n1gVOZouBPOPh7nWwsFD2InbWM+bZnEfp8r9o25p1LpY/ErkNdNFUiQ5cAP4S6FlUdz8O0sW6PL+ehfJRJU8v4FZJKutw43BzfUmn2yyCCQYpsB4cYJUvS7shhwZ1hZGtxJOpaJGtDZv1ja3+F2pOb4tGW/LW4eP5jaFTlpkl0xK6rcJnvVFXkFPrtriOv+n2oDFIYy1M0DLbwrFitZNpTbkXCH4pqg7L7MKRDFKDCDQhNXa0kR4/QKM95J76ry4aLrOCb/zQKdMvy8mKwG52Edy2k3frqlzfXHy4v+L/evPvHzbuR2j33ZC1T/ntxH5aZsKZJinypVjEZdsbsg56Ne5wNtzYrLE2fVabuEkVEzUb77j1shs4jH2s/CJ7FtpP2oYkuoqX3m8gaeVNVReUzhxuklrWXqiXamb6yzxUSgBpGkaVeX7XMsQ2YCC+XD27nbIdt1NdsjNhAirabVFU+GRzlPTZdUD4qXfNi42LWhqwkI8WfGWK8btBDXRFFAJ5qzvZUtrDqjWucAQWgtQHMCfe0SIGj/WSak3cJLwBVnO/tzYua9r5BZMCetcobOMT2PnqGWa+okRnAkqybfEPtEskEp9VI9nwVeb9d3yAgn9BVMyogtaxynPoUeb8WHsUd9O2rN+01GAItA00RxLOf3v4TDVwo+J99sS2tS6px0RvWJGtEa9C6euo9DqYiFbWIXUK7BuWPcn7W9S3mKo3521eb9kvQR5I5MGrKlGqWdV3MKLxz4mLaX9we3m9DILeLt84pLW929zdUtXEZdRkcD7LZlizn+UPx25XexqwrlO5/cOyIv8S3bDrdiscp0nSqJdydUuBczvFnOm3R39Siv5C9MMuudU5T0KeyIutVnZPcS+Yq6sAETpDDluGcZA1TPtVr+Lyvvccsc+u+EFBl0wStb1TK/HF5cna77YDoXRD+0HI4X9yNJe1rre9oQlTq+EjVtkKGKHPJxmR6L7DMTincqUBGO6l5YGxqehg3pY/zQSsLyAtW1VNYBwTEyPrGyoklOGnoreD/orne07gPyvhInF71313tHLu9L8Ix5Zl/GPw0CLX4hIUIiVMM2rxFvFNb4i7e4x/ngBhmcuqrZOxGKPDmxj1xG0R2/Lr7CsY0n/E8ZBvBeJqKX4TEKx4V7bHsgyI4AL/sAE73C6Oig8LeBWElkZa5q2CTI9YddFX/0E8HED4YNd3JKM87ypWs/bbDoTG3/OA16gGZzP2e+JbBJxq5+Sw+HwfQYf++wa6E5sjC9KxtmUl6Mi0LXAZWxFN8eAaC5AoDIorXkZA8HC7YoAEQ5I57bONbNiHBnF8hzcgSltpT2qNXngHZBrIbu7hhnexiYo3eg419OUAARGCD21HcMtvHuMWf7O5Z3LeXvVffsqRBQlYSO725QcoDuFFLldiGvRQoQenArEg9sdVhi+fMFpwta6CHFRmYWQrkybGXdh+7G8UhORSAqLstOKja9n6CLebh4b3E8VuJIznKus67ON2TQ4Zxf8Prp5KAUSZXxQiUiWx4v7HH35Il7sbj7y7iDag0XgRSwexOqMjEr3khmlQVLdhaHK3pDKPHPapEThcYmfrkdDCQEuIChhEs6xYAVoEdat5Z064fY9ySkjn0AQPU3FUjVtK9+DKMb5E/+dbPJmg3iNSdGbM24Y4GrdarEarYVqNMLA4D8DqpG5HZ1G1DxGFyNkomB/VtSTH3VDgOiUxJMXx3LwBKuVafTGqd778mvSq0tmJrcpsFlJ5UKI1630qXj3mrUyuxTGfmxE74gdQOCx0trUfvIcZF9Wo4kfmOcGQTm+UO1gbDMlBUaqWAwbtCYAC7Jd9voTO730z0wZ+oJa+pfmBCQMSQfRoaZqyZBv2+l9+BHxgUHYmbFHDVdNbVDMSNgT9IyeEU4O4hekl3SImULxGKklrI95zRXWJ0nLuFb/OlINdVEqa6DlcCoU6XJv7ojuTIjVL0Igj9/piDC6bwPNgzcn+pHAGG+bdsudyWkkbEaV7oOlV0MzMlXpJ8YNbpjxqBckhK8E8sWbhkuyJDEYs/k9hf2NdhDLD1AiCIZeoe8mWwAjG/N2vnf73YjE6BLYK7o5jw+NjUg4GwzZ0jnSR0UwZvk8n9PyW4CRkbsfFevPfeCUIy/XGq1j3BqRMGjcYByGFXP3h7mlEpxYab+xXMykVCpuD3T7UcMzxJxQaw6w9My1d798z7rtqHifIxkWXt/YR+dWO+EiAQ2jMBt/iGjx1wCA1xvGSfqaX65imIOKe7ac6/LLzPZukL+5806vbQHcCedoMrl4EynSILAt3jiyJ7maXM/cXij0gxmaAitzrFMeN8ixTjnFkjlXH7s1J0Xa0MjnhLT5WfSp1UysgTc54WCYY9uyUSacqFo/a7mY6MGNNwiORYCgKkpybFyF1E3Z7fnWJpi/yAKSLkgMgMjv2xrjGlgzxfVUVTxnbntqGenj1xmDtrtNpJbl77A9KDIzr0Zc8xv1R1+l3Ov7a1+92P5jVjRnSOAo0TQEyyU4q3A/SRLeFaZgN/vf/4lsDo+5MmpHujo3LPjxnTjWRfEVbAkOZo2qf97kcCEVmhF11fM5zaaw8RUXz03g9FBMcNnu0Vohtlwx7QtgOGPWg8wptjbZC0k8yijKRt+W340Js2KIZjAiBA7Rv4kzbbUvvdfb8VNWw5W0ndNRwWxxdxZnoycvS/nJKQWGhpuusu0RolgH4p6XaBOvlf2JbzRyweAAA="
  },
  "ace_context_chunks.py": {
    "sha256": "d7e4b17dea0647d859bbeea5d7809099c5582f7ee2b3bde05aec385864aa862c",
    "gzip_base64": "H4sIAAAAAAAC/6U827LbRnLv5ysg7QOAc4YgD3VZm+dAVV5Fu3ElsVW2k0oVw0INgSGJFQjAuJyLFVXlH/KH+ZL0ZQYYgCDlLetBIgYzPd09fZ+GXr58+d37D7OfG1U626LNE5XM4iJv1FPjqKdSVelR5U3tyDxxjkrWbaUS4dTyWGZqJrN0n6vEqZu0iQ9pvg+urn5Sba1qpzkoJz6o+BO83sq6nidVe3Qei+qTqurA+ZjJfF5WxVY5idrJNmucpnDywknznapUHqvg6gelktrJ22P5DDvGKf2DKO7STAlntzuWai+cMmt5O/WUAiL53nn/7//0nVO1eQO4O7uicmYz9aTitgGgL1++vEqPZVE1jqz2paxqZZ4Psj5k6dY8/r0ucvP7KJvD1a4qjk4Jv2CSo198xBf6d31omzTrntot0Berur4yQ0SKI4Gmsptl6MHhesd7EK1BDbyVmdkoLqpKZbJBwtM8iUolP/WQVabipkrjaNdmWQSI7wkeMCNTcCTf/fLBCZ1Xy8VicfXTjz/+Ag+IuBdFuHUU+UGl6iJ7UJ4fAEvwwNe3m6urKzgbIEsu37z1kHB/deXAnyTdq7oJNb8CPcGnd49pc2DgtCAoSpV7brV1fcKoqZQ8MhT8g4ezzYr4E5y7kzaq8jJ53CZyxRMBLZl4t4vl62v8yxdb1/VXvH3Qlgmww6PlvHelmrbKNXrBQT3xL8CMKXmsYIsIz5WQEw8ya5WmCQfCHu+7RiFnZfUc4nOAZEV1u9ulT54bNMfS5S27aQEDR73xcIcggcOuPdpCwIEBT8OlkFlWPEa5zMO/yqxW/o37X/kJpEqVmYwVI6JRz+Rz0TZeUzQyE45W0PDbhQDRrPZpHr4FuZCJCoM338D7VtXhD0UOwqJyVe2f6UFTSijVoaeBCAYgcDVjku4cT+bPXgryUzcSdNF7ENuiyHwHjisvGmfwxkvzRuyyQjZ+NwH1JUhrkFRgivfg00E/4CHz7n4nAXrB7eI+1Ajdh28XCzO+uEe87hlHHORf97f4Wy+YLa/N6MLvZauSaa2c/8D9PlRVUXkuGhSzCHb8v//5X9ipNix8F97WzETE01toxvjI7krB20XtdgwacYHPBRjh9wRdo9rdh/SKaKKBS/j9XLRVrJxjWzcOGEaDoVMrQDoxu//J+R4o2CvgRZGozEnaSjZpkdeAdpy1gL4CW13KJEGD7PygHmCqeoqReDSTlfoVTqBBVsgyIJBH+ZQe22NIxwZHCchoNvmEtNaQJKzQXHnMGHrj3xF3Qg0BjgKmGSbhm/vbr9Nt6DFHA3g5mZIP4EYysJ+AtcyBG7VTMExHtklaaHbEgFIT3uJ+mtUaFzCKsA88eEtBhMUqzTw+KsbTnyM0X8ORpYzTJgXVWOPwDUxAoClKw3ohaJ/Z7YbBEhoo1PS6kvleeTTD3/TUI2Lh7cqGrHHjSeCRYGixuXPKIktjmmDe0G8CAz9B1BzUYL03jJhNRnranNXT5nfoadOrZa+Zi/vmnng2JzeCNDdIM2JBcPDHi7AGN6QSr1aNR28uKuJfUIzA1DnooetO4BkN4bR5CiKKzh5hUugBdKQg2Si/6N302Y8O4JYPyd5aHSUAzfeshjPk+BoOsZvQgCirJjTjNyzg3bLr/uzWKbyf1+3RG46tNn7PtqxA6fM6cFr4Be8+XjxYekjDY5r3S0c7axAsZLPUv9agewAgDVnx7pCuJgxf4ewgdku3EGdsDe8x1qvGFoE1cK7tbZYe08bYHYtdiOghFUhqVggeHHBBIDX2W1A4wlbopTx6sxzRUJRkycI1n0PDr4dCx4Teh4Mp9+Eh3dis0JBWtkSjwhHu+qX4pJ5DDjicZiW3tdfMDDFaJ59DF6K4B5C9JDKcczuYqI6jHUBYU4xL6jCDaNRjwSSG3NwKRHU+By9FirOeZuNmEuUe8BBrjx086C5SzQ9sJhb+TbBYLK8HdM2H/Ca7runEyITXR7mSVcQr3DFYdysztChJhMrb8wIFN5AlhHsJCrHfGzUzyhv5V+PZJNl6+NDmnzorSOotPCm2Ph6+gvhZVRjz/ZaWpCiCtIVUsD8GMHhVIwAwSeBCyBmaehI8dtNbNO09D4z/DIGVyYxWM5u6GZC+KJ0EsBMxS/wuaOnZQBQY0j67GPo9uatUuOgyIgLvrqR+hDnuaitgVtk25iUToMdoBvwlBmdm/XEpCogMSu5qAs0vg/j4s8vpW4S8dFdkoFxiTbSrJBhkd8WMcjVxEdiiyAQhKxM3Xg2xIIvRz9JRpYvxVD+KT8I1ihQZCXFX5hfsSQyEbejfLzr+xUQgIrc/SERkI0VdhfWOMwWK6mX2KJ/raJmEv1TgRZLmuVShS47v1bKP4OrqRUgeDYQM4WAaUypIe16ES+P78rJ3jjjHDyCA974eY/ISkERIZQvIuz7982+DmMUkKgBS04d+GhJkyBKiYudB/J/rXfoXIY4GoJKe2w+6oh8tIYkDhRWfv/in01xMwl2/I79/x5ECRLw8RbiYqcPUCT/yc1uW7JPpdCwg9YqyfOHoxTaZ/SyTh1EVIAL19vp3hl4btTBknFYakk5op+bxvv3J6MQYhyOqM1gxATN/+OYPEctkYS5dE1XW+endPLffL8D80PA5eJQP5jdl764/pK2Th45ItsOey9yISkDFgOzqAIifBj0Y65AFYcFM36sp5xAOZJdgbn5faklVA51fXgv7HLSUMel8zOH0adMMvTnvzdk3D1kFCUHDPMUapvU8Ghw/JSkwnQsXrPVUC4qKTzrP5tlVCuSAlcPCRairFno/mrBHP0cWMyLjGzL8uce22L3hyQHXAQJwmKrytEpx8SeIi/IZuWP4OgbZmx/e/uT9i3CA5iVT86Oe6NBKkFKMNRLn8YDlJD5frIZhhIcCZPIlyKxyQ5mrKUJJ0cnle5kXeRoDWLJYmHqgH0YIGBHPj+kTQi3y7Dlw/sboQ+iOdSvaCY1Q9aASDc5QA2dNOHFJDCGAn4oPulKIsRKCmoHfhHALS4ok3DMUboe0puZUtS+rBVWbe2uXy4Ag57O8qBvIefEnyT1yicbxr9QVIHvmuGFAVjzCYemkd4VJMaxd4vp4JeGfMj5G9e3bTPFa4qW/EVToJMFjHhPnQstn8cQ+tUPd6fJEo1H9WR9VI9E5hFRJAs+V1B6pQTfXJ4fH5SZ/kAOYtewCmN6Ipc0F6RqK/TB4vZCoGahOUsCR6dQxPtDhWUdM9QuUWytnIIKHWCXFY75VElILsd74NyOUwVxhdE7vhlBOkkyOApSJUD2KXyyqqDgY0hmsORqkGTMOxL/xVxgVZir3aIrPb2/M283YZ1AQ4UFUUP9a0b9HOFauQF5fL32ND6pKqAt2FuyJeptAKgTj3q81lc0axOooowdV1RieLkHkZNPWkJGgLqrEFZ2p4FMdmg6hz56Mta3op8I+kBIjIiyzgqMKbcWDjz99+PnDL8Iy+f3PU7iWdbORGRu9IAeui9PJk1SdbALCCPT9oK8OKoX3FBB4zTshC5z3VmFJF8lq7d1qXV7gCtSgcvY9hO4YxWOMxNHoHbnHNG/T5plvRNqaTOWvrcxwrM3htNJdqpJAa4BVczY2l86YHTY5djtOwAHtoU/44e2KDFJ2YUUXVLXW4/M+EpzmO2+3du0hyJxGPhUvfWLQsvASOG1PhgD14GZQJC25whTR5UX6oKKm0Pj2+1I2bF8zvAgNFpf83986tIzlMS4QwKkaDlt7F3AUw4gUNzGFdTCuEZJh5xWjy4Ae0zvSbcsqE3knppjR4ZsDDkvmE6y32aTVs7+GGbHMAmVzS5vxF+HkQVxiXu/mfy/vAFPeRecZkChNhnZdwOpra0H4kjsPOL8coz3IPS9i/ZG2JoCMtUlxnSTdgfbXHIhY4c6pFlmsnNC/0UttRLWwkHI/R2QMbG0UNNJF/Jg20BmDxVZH/hWFo/TAmmsLlFFmC8qEr0e4oZ7Z7XHXhQ282CSBGLpiUthd7tiv2adgZGDUHBwLMtiepE+8OzAjAfYtztCJjzyTWfoH7MnkNlwxAeB0AOtBHWUzLOKcrO2qnn3tBAB5A0hYfdnMpoD7X4GuadjLlMAGfz6dMirc9FSMXgzYPFIkOPsx9edxHsDRBoQBDCUC5Mkc28Vax3vKiylgz7EkOAehwAv4em5wn6PLdo5pTdHi0AjjPqa60+asVLVXFQVrXmTZY1vVtG217HY3+atZZ2+a7mwtjc5op97qsiO+sqoNuzhvsv4a3OiyG/QgAgwWXXMvLm/4XhwHrVtxBIP3cPEnD/8SPPCvP77/l+jDf/639fTDX3rlgHPYg17q7NxsbUY54DiZPPBlNoCBzSHbP3hL+XXt+VyL+PxlcKNv6iXaX+iK3maYbGAVm3IoI6hYJ92AhyXja/Df8eroM097SdNeblaLV8kXd1jGpnW6FtCl/31OZqVIsDVhpylanYSUsYTQO+kDKz1xDQs343hpBBudOS8/deHkvVeTzhgXkPPpnC8R7J7uMfBBvJPtg07mlxUEt55LbTmOiU41dLxKELusrQ/MqDsd3qoBlD85P6m4wFtkSZXNGnGVDTYsNI7cNfgipxC6qtoSC2aGYRz9BsMLjQJrzxhgdkmwLTsGbprr6z/Nt2CfFVtQGH4dXQ9MU5+mkTvVk2wfOnVU2uilLNIgyxjRW/qg39uKMPSZoe0zV5MlhMFpabzs47rr2aHf3m1hw09jke2mmavg0+20Z2DnUT43B0g5KIZFkwo6DOe4k2nmbNUOL69j2Ia6peDscN9nw/rTBKs9bsGiLk5ecKVJHxCoqjmcz7yCldTvbMWKh2/C21OF67hwGdgpCpyWmmNg96Frbx1QQf7vfIwwXf3p74X+oZBA6FazCdNzqr4dhuc0eGB62F6aNcFpSgV+ze5xsi22ME/DcFcPaj+cbD27J2pQ9VgurvMSNGN/u/CwlMKjPF3cqtkt1T+0Pz/WuuoxAWhcPqHnFmvRtERf2MAoTX/72u/hlthmOIB8LPKC6zsMTIKshbf+erX6ZhPIGkF5FigTAmNZBpf6929fU7ALCPMzUPJ61V2UHX5zV1TndrETskjSGLJ8d7UIvnS7z0L8m3fX1b847Lr1CKrQfyV4D4UlfgGB4KFI4GkH0eu6wwa7CTSIOXJAd4t4t6KfAsNS3yqbMZ0UZMWjOKT7g74Zf71YLObLNwu++rTGXr3xCZ6MYUNddsJuQoqCTGshvF0DwBUCRK+MYzcQdT3aWStCoTeUk8KKxeY+XPxu9m2xi5CFAtYSoA3SDIb2LpP7EB0X4l5yBZp8Nk6ixhdYsHkXIojr4Ns3pomnyHdpgu2jYSdscZaWRI3cb+aEo1iIW390JYqoEndgGhtcA+ldGLx5w3HOBC39PHNVSTeejwr4BkbPbkiRaJ/xWDOISEvs8FsIeCrT+ZIbV05F38YRSSngXBAKCLhAZDYIAAz/aNR0bip5jB7hSItHEJZdIypESmRkZ0TF/8QKHbd4TJMu3Nbmkl7M6MV8vrzDG3V6c0NDTFKIcNc0PMumjCLeXU+/YUnfhoSUBlGdBVGdB4HSwLe3kL4RbmJJArk9GZ240/wrNvyBd4dNapDI5wKCgG0BgZiu15rLzEcpHrfh4HCZZ3fbDFkjrx/lzfb6cctVNCkUkIZ20Bf4zxZTjh132JH585S8VvJGba/V1p8vjfxWVS+5bNEmzNg1rKD1W47NUUloR/8d2K8/98KqrzbBjYVg2wkPRNaH0Gjncwy0s5YsAn1r8xHGIaz7tU0V3Qlqa5Y4DZieOuXuGVBGyPDT35TmF6aAWcblGAyltm2aJa1uMQTRbUKPCMa+FG5LIcz8Gw+fvdsZ7uPrxsEKAx20MX3f22KUPuo2D0Ekic8uC6zVqEDP7JRdlNMIGBAl2x28A24oaV37uCSFowlbn9o02M9iawU+CJf260N7XuOuiBgLYlkpMBJYX8jUA4RkXGCjqQR8jpYOTk37TtyqbtwVnz7+9o1F4fb6SFaVfK4pPxXk8mqtr7prZpht3ZF0al/Arwa9GBur9Up7RF7qvwi7W4t6SmM+QAzzrJM807pHFaG8ce17WTRkv6kKbJY3VecDdRwZvFfLvq2PjZO+gswd6vgh9IakG/2nCMBo+++M17SVGDd6cHgx7vS4VAFhLGn3eY7+gLo/RlUPMpdia6JQqxtoI+wxwrev1BAn13K13ehrLDlJy2o7O28g0Q8Mm6rsnknr5C16e3/BGOsuRPN7c5cXRyzwafM9oKdvxVIS7LbpGgYjdRu80TW+Ub/QZv7KH/UCWq103lrvtumr0KdNRBtCMbzYsWf/IaPCXNDgZ4yw0I83/Ng17i0X1lUrZsWNBD5KjIPARjsHme34uxd93WS61tuma16l9nJYoSBxd1Syt5LjpkqPA3Uds4j4YzzFiEPrmLWGmiS7YbKlnQd/Z85qICI3uO8kf9CGxzfs5Tog/j27/IF2IYhB9yVXQQwa0x2pKJVcJ6N+CjjCvCGWOElax7JKLMWhSBEFyLNjGrYELJjm90ZYkU6sw5op3vh2H+XT6mn9arNmE7wZKKx2L5jz1xRuWhLwPbhe7FYQOoDQpw8eI9ZXhm4N59pSby0VSFLsZC1oXq4eSRyCsbIPhXElteYzoaO3M/swjC2QU6MnNmW7Gok58arfbLPezqYEZnLZ5MxpM0b8tE4WqxEDCk3b5gQR+uL+lf81Gqks9XVSpod1A8CrgUhf6sgauomsDKsyPJv98BDeWrsr9yN6dvtWWUc9knrLZvuqAPkyH965X4adx1kpYCNOipmPvtApMpJvGQpqjLpdLhbXFIBCFr/0qnKNKG7mmf5BoZd5IO03U0YBpc4Rmtp0KlBVNkyN3TQmK9TPHIJx9IEc5hAglDyEwRQPbM80uVLsRoSFSDJFavxYlUywCa6YUPr7DKyjzFvAsFIPqXoMPSJjfSak2zj3zuwVpbbb2kw9H9Zt/Hdvz3XpDv6gMaKPKu0QF/mN29AbiOMxcT9DAjcdaBoivMdKASlufJtewj0SP/MXhTtIcWsnKWhvhRUc7OBiUZs36XFbqcD5gC0NsxJb3Pqen32LpYdGQbx3kLWy5DawjDWFG6YDmrjWFxms5Aa0HGsZQLJu7tPOFa/MQg4WBGT2c1zGNxDwgyXxNrAzAV5+jesgC6Dd3RX9I1ysGmPniJIJ0Hfk+7gVTcWTpGcqeLgr/Ptsl7VjFFbzMEnlHvvN0rimVrg7/HZVtk0B+pXGTpzJ9EifKx2KCs41jttKxs/CYfYi42pqeYuf6V5w2EYSuKPIHzvd/sid2PCyiy74sOVm8tJr6mbIm75NmriYZosLseaduWqw486vXgwBoae3LhO3RJO3MDruABjTDSeXL5OmMh19wWlfy/T9EN09/KCkC+8ni7nMGKMVVmsg3cD6HbvMDALE7+yUyhwEC4ZKXJO/6VbYrtY+0c+sl+ip/Ll0fdd1bDppY+4GJPDxAIrPu7jW0Qp9xXI+JT1tzNWXqTv+RNa0Qe3cz50gfuG+NN6BnEXdbrlv/+P7f4uWr91pGO4xfaKmaV4fvLn2gj9f28zlRpMb3YJ3ATSKJ+JDX/t1iIluB1tK/2AP6pABTLvftZhm6fZYvoK4VOGSLY3dfrv8NL0WproTDahEDcbXmOfRia21YbTI4PqITjnw7XpcQtmMUrLTM6Rw/DMtHqiTdaA6ntPlxFcMctx4yW9v9NvNpZPqDp+VmcmrwRjZ0taXEHpFMnWEofERRvPGnxDpz4e8iYT9wvWP9RWcyvfNgTzZN8EQzKAmYQMew5joHhxSjZIwYc10e+HAm33mpfrLGmH3eUSGB+A3rb6QkTdEh0kRB36BqSO5vth2SszU3VoPwyxkNgm7KYiJc1e2B4JTPN8T9cX/w7qJX75Qgzh9/2U1jJ8ywp01/I5R/4dV97wTitj+Rmxdzym3vpc2XcK6KbjzCNPNuV3SItQTc5HKcKGlgxOxT9dbFk6Ya0uATM/wWdvun+2G5WiR5ZUpG5R5eUiHQ/hVrinKlaH5/zuC76o90fgRnyovAV9dpfSJZxhFSRFHkX8HchFCtJkkEUoITaxxZhOCzB4hK0hweyuStnq5YQWt5GWey+2F1AtK41Lv77mzGXscV5Dhwuv3EdTpVQW26J1b0jeGj5eZ/wDBKBODoBBb6P9WJfx2cWZPLi9dXPv2wt6Yu11cHLz55szO2PVukXthE/sLNgil0xhLXcPv1Dbdhvrjo87+o/SRO3fpAxJYgcGb0JoycOjjI8alIDMn+NDJXz4oE4kChBdGK8GsnoDCTgUblKEC/3+Wqa11WwE4Vfo8AkGjqQD2KPdkZy6CAs0TG3d9hbw5/q8AHWvfLsS3C3G7XGzGYnsKh6381MnrixqYXIPOEUtxZe11tw34FGi9C/mDBtf+ar9GMOYTNZqsezroN2iL0BD4+1Qthjw4LJvymH3zoZfilxeWpaTB8fdpKpvAlflqNbxdyMNobZ9b9X1qfX8bwYs+00xzNJyR3P2eljYuvfS9OWd7cmjiSUsOjdodOZr3X+vs7hpuCG/qurncJikuYqQ7dPik6WKv674ZMKZrrqFR/XD+rFDfh/dE1rKVJtVqPu2JmTg4KrlFgleJ6Pwp00QN3ErgR7O4L8/6r4I0YPN/BVHTC6AckReNIiAnitABRhF+XY2O8Or/AW5E1kW9SwAA"
  }
}
# END EMBEDDED CONTEXT HELPERS
TOOLS = Path(tempfile.mkdtemp(prefix="drum_chunks_"))
for name, payload in EMBEDDED_HELPERS.items():
    data = gzip.decompress(base64.b64decode(payload["gzip_base64"], validate=True))
    assert hashlib.sha256(data).hexdigest() == payload["sha256"], f"Helper checksum mismatch: {name}"
    (TOOLS / name).write_bytes(data)
modules = {}
for name in ("electric_full_song", "drum_probe", "ace_context_chunks"):
    spec = importlib.util.spec_from_file_location(name, TOOLS / (name + ".py"))
    module = importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
    sys.modules[name] = module; modules[name] = module
context = modules["ace_context_chunks"]
ROOT = modules["electric_full_song"].discover_root(Path.cwd(), globals().get("ROOT"))
if ROOT is None and Path('/content').is_dir():
    ROOT = modules["electric_full_song"].discover_root(Path('/content'))
if ROOT is None:
    raise RuntimeError("Source bridge belum ada. Pakai VM aktif atau sel setup notebook 12 sampai bobot siap")
if not DRY_RUN:
    modules["electric_full_song"].model_python(ROOT)
SOURCE_PATH = globals().get("SOURCE_PATH")
if SOURCE_PATH is None or not Path(SOURCE_PATH).is_file():
    uploaded = files.upload()
    audios = [n for n in uploaded if Path(n).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}]
    assert len(audios) == 1, "Upload satu piano asli seluruh lagu"
    SOURCE_PATH = TOOLS / ("piano" + Path(audios[0]).suffix.lower())
    SOURCE_PATH.write_bytes(uploaded[audios[0]])
DRUM_CHUNK_OUT = globals().get("DRUM_CHUNK_OUT")
if DRUM_CHUNK_OUT is None:
    DRUM_CHUNK_OUT = ROOT / "runs" / ("drum_chunks90_" + uuid.uuid4().hex[:10])
    drum_plan = context.prepare(SOURCE_PATH, DRUM_CHUNK_OUT, context=90, margin=6, fade=.58, instrument="drums")
else:
    DRUM_CHUNK_OUT = Path(DRUM_CHUNK_OUT)
    _, drum_plan, _ = context.load_plan(DRUM_CHUNK_OUT / "plan.json")
    assert context.instrument_of(drum_plan) == "drums" and drum_plan["context_cap_seconds"] == 90
    assert drum_plan["original_sha256"] == context.sha256(SOURCE_PATH)
DRUM_CHUNK_PLAN = DRUM_CHUNK_OUT / "plan.json"
print("Runtime dipakai kembali:", ROOT)
print("Plan drum:", DRUM_CHUNK_PLAN)
for c in drum_plan["chunks"]:
    print(c["index"] + 1, "model", c["model_duration"], "detik; core", round(c["core_start"]/32000, 3), "sampai", round(c["core_end"]/32000, 3))


## Generasi tiga konteks dan penyambungan

Jalankan sel ini lagi jika satu chunk gagal dan VM masih hidup: chunk selesai dipakai kembali setelah verifikasi. Tiap chunk memuat model; margin/reload menambah waktu. Tidak ada fallback otomatis ke durasi/seed lain.


In [ ]:
from IPython.display import Audio, display
if DRY_RUN:
    print("DRY RUN: plan saja; tidak menjalankan model atau stitching")
else:
    progress = context.run_chunks(ROOT, DRUM_CHUNK_PLAN)
    finished = DRUM_CHUNK_OUT / "stitched/seams.json"
    if finished.is_file():
        drum_seams = json.loads(finished.read_text())
        assert context.sha256(DRUM_CHUNK_OUT / "stitched/drums_full.wav") == drum_seams["stem_sha256"]
    else:
        drum_seams = context.stitch_run(DRUM_CHUNK_PLAN)
    print("Sambungan:", [(s["index"], round(s["center_seconds"], 3)) for s in drum_seams["seams"]])
    print("Seluruh sambungan perlu didengar; flag numerik bukan penilaian groove.")
    for name in ("drums_full.mp3", "mix_full.mp3"):
        display(Audio(filename=str(DRUM_CHUNK_OUT / "stitched" / name), normalize=False))


## Simpan hasil dan kirim feedback

Paket feedback berisi mix/drum penuh MP3, cuplikan tiap chunk sebelum stitching, cuplikan sambungan, serta report aktual tiap generasi. Paket result juga menyimpan WAV drum tiap chunk dan WAV hasil akhir. Keduanya tidak menyertakan bobot model atau piano asli.


In [ ]:
stitched = DRUM_CHUNK_OUT / "stitched"
if (stitched / "seams.json").is_file():
    for filename, full_result in [("drum_chunked_feedback.zip", False), ("drum_chunked_result.zip", True)]:
        bundle = DRUM_CHUNK_OUT / filename
        with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            archive.write(DRUM_CHUNK_PLAN, "plan.json")
            archive.write(DRUM_CHUNK_OUT / "progress.json", "progress.json")
            for path in sorted(stitched.rglob("*")):
                if path.is_file() and (path.suffix in {".json", ".mp3"} or path.name.startswith("seam_") or full_result and path.suffix == ".wav"):
                    archive.write(path, "stitched/" + str(path.relative_to(stitched)))
            if full_result:
                saved = json.loads((DRUM_CHUNK_OUT / "progress.json").read_text())
                for c in drum_plan["chunks"]:
                    folder = (DRUM_CHUNK_OUT / saved[str(c["index"])]).resolve()
                    assert folder.is_relative_to(DRUM_CHUNK_OUT / f'chunk_{c["index"]:03d}')
                    stem = context.verify_chunk(folder, drum_plan, c)
                    archive.write(stem, f'chunks/chunk_{c["index"]:03d}_drums.wav')
        print(filename, round(bundle.stat().st_size/1024**2, 2), "MiB")
        files.download(str(bundle))
else:
    print("Belum ada hasil lengkap. Tidak ada audio hasil yang dibuat oleh dry-run.")
